# System 1 Notebook 03 - Merge, Index, DB, Validate, Smoke

Notebook này chạy release flow đúng thứ tự:
`merge -> build-index -> build-db -> validate -> smoke-test`.

Notes for Kaggle/Colab operators:
- GitHub repo bootstrap uses `git clone`, `git pull`, or existing `AIC_REPO_ROOT`.
- Python package install uses `pip install -e`.
- Runtime paths support `AIC_REPO_PARENT`, `AIC_DATA_ROOT`, `AIC_RUNTIME_ROOT`, and `AIC_ARTIFACT_ROOT`.
- Set `worker_id`, `batch_id`, `execution_mode`, and `provider_mode` before running worker cells.


## 0. Bootstrap repo và cài package

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path


def detect_platform() -> str:
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    if Path("/content").exists():
        return "colab"
    return "local"


PLATFORM = detect_platform()
print(f"platform={PLATFORM}")
if PLATFORM == "colab":
    print("Optional Drive mount:")
    print("from google.colab import drive")
    print("drive.mount('/content/drive')")

GITHUB_REPO_URL = os.environ.get(
    "GITHUB_REPO_URL",
    "https://github.com/awun0105/Multimodal-Agentic-Retrieval-Engine.git",
)
if PLATFORM == "kaggle":
    default_repo_parent = Path("/kaggle/working")
elif PLATFORM == "colab":
    default_repo_parent = Path("/content")
else:
    default_repo_parent = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

AIC_REPO_PARENT = Path(os.environ.get("AIC_REPO_PARENT", str(default_repo_parent))).expanduser().resolve()
AIC_REPO_ROOT = Path(os.environ.get("AIC_REPO_ROOT", str(AIC_REPO_PARENT / "Multimodal-Agentic-Retrieval-Engine"))).expanduser().resolve()
SYSTEM1_ROOT = AIC_REPO_ROOT / "system1"


def run_shell(command: list[str], cwd: Path | None = None) -> None:
    printable = " ".join(command)
    print(f"$ {printable}")
    subprocess.run(command, cwd=str(cwd) if cwd else None, check=True)


if not AIC_REPO_ROOT.exists():
    run_shell(["git", "clone", GITHUB_REPO_URL, str(AIC_REPO_ROOT)])
else:
    print(f"Reusing repo: {AIC_REPO_ROOT}")
    run_shell(["git", "-C", str(AIC_REPO_ROOT), "pull", "--ff-only"])

run_shell([sys.executable, "-m", "pip", "install", "-e", str(SYSTEM1_ROOT)])
print(f"repo_root={AIC_REPO_ROOT}")
print(f"system1_root={SYSTEM1_ROOT}")

## 1. Helpers, platform defaults, và path runtime


In [ ]:
from __future__ import annotations

import csv
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import JSON, display


if PLATFORM == "kaggle":
    default_data_root = Path(os.environ.get("AIC_DATA_ROOT", "/kaggle/working/system1_input"))
    default_output_root = Path(os.environ.get("AIC_OUTPUT_ROOT", "/kaggle/working/system1_output"))
elif PLATFORM == "colab":
    default_data_root = Path(os.environ.get("AIC_DATA_ROOT", "/content/system1_input"))
    default_output_root = Path(os.environ.get("AIC_OUTPUT_ROOT", "/content/system1_output"))
else:
    default_data_root = Path(os.environ.get("AIC_DATA_ROOT", SYSTEM1_ROOT / "input"))
    default_output_root = Path(os.environ.get("AIC_OUTPUT_ROOT", SYSTEM1_ROOT / "output"))

AIC_DATA_ROOT = default_data_root.expanduser().resolve()
AIC_RUNTIME_ROOT = Path(os.environ.get("AIC_RUNTIME_ROOT", str(default_output_root))).expanduser().resolve()
AIC_ARTIFACT_ROOT = Path(os.environ.get("AIC_ARTIFACT_ROOT", str(AIC_RUNTIME_ROOT))).expanduser().resolve()
AIC_OUTPUT_ROOT = Path(os.environ.get("AIC_OUTPUT_ROOT", str(AIC_RUNTIME_ROOT))).expanduser().resolve()
CONFIG_ROOT = Path(os.environ.get("AIC_CONFIG_ROOT", SYSTEM1_ROOT / "configs")).expanduser().resolve()
RELEASE_ID = os.environ.get("AIC_RELEASE_ID", "competition_dataset_v001")
RELEASE_DIR = AIC_OUTPUT_ROOT / RELEASE_ID

# Current CLI writes artifacts under output_dir / competition_dataset_v001 / artifacts.
# AIC_ARTIFACT_ROOT is reserved for future split-storage support.
AIC_DATA_ROOT.mkdir(parents=True, exist_ok=True)
AIC_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
AIC_RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)


def cli_args(args: list[str]) -> list[str]:
    return [sys.executable, "-m", "system1.cli", *args]


def run_cli(args: list[str]) -> None:
    printable = "system1 " + " ".join(args)
    print(f"$ {printable}")
    subprocess.run(cli_args(args), cwd=str(SYSTEM1_ROOT), check=True)


def load_json(path: Path):
    if not path.exists():
        print(f"missing: {path}")
        return None
    return json.loads(path.read_text(encoding="utf-8"))


def show_json(path: Path, title: str | None = None):
    data = load_json(path)
    if title:
        print(title)
    if data is not None:
        display(JSON(data))
    return data


def show_jsonl_summary(path: Path, limit: int = 5):
    if not path.exists():
        print(f"missing: {path}")
        return []
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    print(f"rows={len(rows)} from {path}")
    for row in rows[:limit]:
        display(JSON(row))
    return rows


print({
    "repo_root": str(AIC_REPO_ROOT),
    "config_root": str(CONFIG_ROOT),
    "input_dir": str(AIC_DATA_ROOT),
    "output_dir": str(AIC_OUTPUT_ROOT),
    "runtime_root": str(AIC_RUNTIME_ROOT),
    "artifact_root": str(AIC_ARTIFACT_ROOT),
    "release_dir": str(RELEASE_DIR),
})

## 2. Cấu hình user-editable variables


In [ ]:
worker_id = os.environ.get("AIC_WORKER_ID", "worker_release")
batch_id = os.environ.get("AIC_BATCH_ID", "batch_000")
execution_mode = os.environ.get("AIC_EXECUTION_MODE", "debug_small_sample")
provider_mode = os.environ.get("AIC_PROVIDER_MODE", "mock")
num_batches = int(os.environ.get("AIC_NUM_BATCHES", "1"))
input_dir = Path(os.environ.get("AIC_DATA_ROOT", str(AIC_DATA_ROOT))).expanduser().resolve()
output_dir = Path(os.environ.get("AIC_OUTPUT_ROOT", str(AIC_OUTPUT_ROOT))).expanduser().resolve()

print({
    "worker_id": worker_id,
    "batch_id": batch_id,
    "execution_mode": execution_mode,
    "provider_mode": provider_mode,
    "num_batches": num_batches,
    "input_dir": str(input_dir),
    "output_dir": str(output_dir),
})

package_release = os.environ.get("AIC_PACKAGE_RELEASE", "0") == "1"
print({"package_release": package_release})

## 3. Run merge -> build-index -> build-db -> validate -> smoke-test

In [ ]:
run_cli(["merge", "--mode", execution_mode, "--output", str(output_dir)])
run_cli(["build-index", "--mode", execution_mode, "--output", str(output_dir)])
run_cli(["build-db", "--mode", execution_mode, "--output", str(output_dir)])
run_cli(["validate", "--mode", execution_mode, "--output", str(output_dir)])
run_cli(["smoke-test", "--release", str(RELEASE_DIR)])

if package_release:
    run_cli(["release", "--mode", execution_mode, "--output", str(output_dir)])
else:
    print("Skip package release. Set AIC_PACKAGE_RELEASE=1 to enable.")

## 4. Inspect release reports

In [ ]:
show_json(RELEASE_DIR / "manifests" / "merge_report.json", "merge_report.json")
show_json(RELEASE_DIR / "indexes" / "index_version.json", "index_version.json")
show_json(RELEASE_DIR / "manifests" / "validation_report.json", "validation_report.json")
show_json(RELEASE_DIR / "manifests" / "smoke_test_report.json", "smoke_test_report.json")
show_json(RELEASE_DIR / "manifests" / "dataset_manifest.json", "dataset_manifest.json")